## Data Fetched from API

* In this notebook, we will study a mini project like topic- Data fetched from API or Web Scraping.
For this, we will use the API of TMDB.org and database of Top-rated movies from the website in the JSON format.

In [4]:
import pandas as pd

In [ ]:
import requests, time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

API_KEY = "7da4c5c53d9289d717de7f959a79aaa0"
BASE_URL = "https://api.themoviedb.org/3/movie/top_rated"

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (compatible; MovieDataApp/1.0)"})

retries = Retry(
    total=5,
    backoff_factor=1,                     
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)
session.mount("https://", HTTPAdapter(max_retries=retries, pool_maxsize=10))

all_movies = []
for page in range(1, 501):
    try:
        resp = session.get(BASE_URL, params={"api_key": API_KEY, "page": page}, timeout=10)
        resp.raise_for_status()
        all_movies.extend(resp.json()["results"])
    except requests.exceptions.RequestException as e:
        print(f"Page {page} failed: {e}")
        continue
    time.sleep(0.05)  

In [14]:
import requests, time
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

API_KEY = "7da4c5c53d9289d717de7f959a79aaa0"
BASE_URL = "https://api.themoviedb.org/3/movie/top_rated"

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (compatible; MovieDataApp/1.0)"})

retries = Retry(
    total=5,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)
session.mount("https://", HTTPAdapter(max_retries=retries, pool_maxsize=10))

# Page 1 tells us how many pages actually exist
first_page = session.get(BASE_URL, params={"api_key": API_KEY, "page": 1}, timeout=10).json()
total_pages = min(first_page["total_pages"], 500)   # TMDB hard-caps at 500 regardless of what total_pages says

all_movies = first_page["results"]   # already fetched, don't redo page 1

for page in range(2, total_pages + 1):
    try:
        resp = session.get(BASE_URL, params={"api_key": API_KEY, "page": page}, timeout=10)
        resp.raise_for_status()
        all_movies.extend(resp.json()["results"])
    except requests.exceptions.RequestException as e:
        print(f"Page {page} failed: {e}")
        continue
    time.sleep(0.05)

    if page % 50 == 0:
        print(f"Fetched {page}/{total_pages} pages...")

df = pd.DataFrame(all_movies)
print(df.shape)
df.head()

Fetched 50/500 pages...
Fetched 100/500 pages...
Fetched 150/500 pages...
Fetched 200/500 pages...
Fetched 250/500 pages...
Fetched 300/500 pages...
Fetched 350/500 pages...
Fetched 400/500 pages...
Fetched 450/500 pages...
Fetched 500/500 pages...
(10000, 15)


,adult,backdrop_path,genre_ids,id,title,original_language,original_title,overview,popularity,poster_path,release_date,softcore,video,vote_average,vote_count
0,False,/ezbrL1dMymKQZw7mDEWa2ZTzN7d.jpg,"[16, 28, 12, 14]",980431,Avatar Aang: The Last Airbender,en,Avatar Aang: The Last Airbender,"Avatar Aang, the world's last Airbender, learn...",119.6582,/3sgnSfNT27Bx5O5ukr7B26mhEQq.jpg,2026-07-24,False,False,9.200,886
1,False,/iCML6fBe3X0y06P33bM7dyYXPKZ.jpg,"[35, 10749]",1632181,Accidental Partners,es,Socias por accidente,Two women discover they were both scammed by t...,25.5382,/j0CIVzeR7hRAPBPGR54qDZoOQpp.jpg,2026-03-12,False,False,8.910,354
2,False,/zMwhWailP1WY7sb6AoE6b8ugoy.jpg,"[12, 16, 10751, 14]",1007757,Swapped,en,Swapped,"A small woodland creature and a majestic bird,...",51.1576,/tHhxWxge06goXU6ZQH1hj7vK8Hd.jpg,2026-05-01,False,False,8.880,2109
3,False,/zfbjgQE1uSd9wiPTX4VzsLi0rGG.jpg,"[18, 80]",278,The Shawshank Redemption,en,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,87.0115,/9cqNxx0GxF0bflZmeSMuL5tnGzr.jpg,1994-09-23,False,False,8.700,31064
4,False,/1RgPyOhN4DRs225BGTlHJqCudII.jpg,"[16, 28, 14]",1311031,Demon Slayer: Kimetsu no Yaiba Infinity Castle,ja,劇場版「鬼滅の刃」無限城編 第一章 猗窩座再来,The Demon Slayer Corps are drawn into the Infi...,97.2507,/fWVSwgjpT2D78VUh6X8UBd2rorW.jpg,2025-07-18,False,False,8.725,1869


In [15]:
df=df[['id','title','original_language','overview', 'popularity', 'release_date', 'vote_average', 'vote_count']]

In [16]:
df.head()

,id,title,original_language,overview,popularity,release_date,vote_average,vote_count
0,980431,Avatar Aang: The Last Airbender,en,"Avatar Aang, the world's last Airbender, learn...",119.6582,2026-07-24,9.200,886
1,1632181,Accidental Partners,es,Two women discover they were both scammed by t...,25.5382,2026-03-12,8.910,354
2,1007757,Swapped,en,"A small woodland creature and a majestic bird,...",51.1576,2026-05-01,8.880,2109
3,278,The Shawshank Redemption,en,Imprisoned in the 1940s for the double murder ...,87.0115,1994-09-23,8.700,31064
4,1311031,Demon Slayer: Kimetsu no Yaiba Infinity Castle,ja,The Demon Slayer Corps are drawn into the Infi...,97.2507,2025-07-18,8.725,1869


In [17]:
df.to_csv('movies.csv')